# Assignment 3 — Traffic Sign Detection & Recognition on Roads

**PyTorch custom CNN + YOLO sign detector + Streamlit localhost website**

**Task:** Given a full road photograph, locate each road sign with a bounding box and predict its sign name. The system also includes annotated red and green traffic lights from the second dataset.

**Two-stage architecture:** Full image → YOLO detector → bounding boxes → cropped images → **custom PyTorch CNN** → traffic sign names. This separation is important: *classification alone cannot find signs on a whole road photo*.

**Submission:** Jupyter Notebook (`.ipynb`). Defense: under **4 minutes**, demonstrate your actual code, plots and running application. Do not run full model training during the defense.

**Data sources:** [German Traffic Sign Detection Benchmark](https://benchmark.ini.rub.de/gtsdb_dataset.html) (900 full-road images), [Self-Driving Cars Roboflow YOLO dataset](https://universe.roboflow.com/selfdriving-car-qtywx/self-driving-cars-lfjou/dataset/6) (4,969 images). **Total: 5,869 road images**. Roboflow `README.dataset.txt` says CC BY 4.0. The third large dataset is optional.

## 1. Imports and reproducibility

Inputs are three-channel **RGB image pixels**, with bounding boxes normalized to `xc, yc, width, height`. The custom CNN's target is the **sign class**. The detector's target is **one class** (`traffic_control_object`) and a bounding box. Preprocessing resizes cropped images to 64×64, converts them to floating-point PyTorch tensors, and normalizes channels to approximately `[−1, 1]` using mean/std 0.5. We use a fixed random seed for the GTSDB training/validation split.

In [ ]:
import json, random
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from torch import nn
from torch.utils.data import DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = Path('data')
GTSDB_ZIP = Path('archive (2).zip')
ROBOFLOW_ZIP = Path('archive (1)(1).zip')
EXTRA_ROOT = None  # e.g. r'C:\datasets\third_dataset' if already extracted in YOLO format
print('PyTorch:', torch.__version__, '| Device:', DEVICE)
print('Working directory:', Path.cwd())

## 2. Load the datasets, validate class mapping, and make train/validation/test splits

GTSDB is supplied with 600 original training images and 300 held-out test images. We split its original 600 training images into **approximately 80% train / 20% validation** (seed 42), ensuring singleton sign classes stay in training. The Roboflow dataset already supplies 3,530 train / 801 validation / 638 test images; its original test split remains untouched.

**Critical:** GTSDB and Roboflow use different class-ID numbers. We map IDs to sign *names*, then merge matching labels, rather than simply concatenating class IDs. Unknown GTSDB class ID `100` occurs once in the **test** labels; its bounding box is retained for detection evaluation but not included in the classification benchmark. A category with no training crop cannot be learned and is excluded from CNN output classes. Missing label files indicate sign-free road images and are useful **negative images** for the detector.

In [ ]:
from src.preparation import prepare_datasets
if not (DATA_DIR / 'summary.json').exists():
    assert GTSDB_ZIP.exists(), f'Copy {GTSDB_ZIP} into the project folder.'
    assert ROBOFLOW_ZIP.exists(), f'Copy {ROBOFLOW_ZIP} into the project folder.'
    prepare_datasets(GTSDB_ZIP, ROBOFLOW_ZIP, DATA_DIR, EXTRA_ROOT)

summary = json.loads((DATA_DIR / 'summary.json').read_text(encoding='utf8'))
class_names = json.loads((DATA_DIR / 'classes.json').read_text(encoding='utf8'))
print('Split statistics:')
display(pd.DataFrame(summary['stats']).T)
print('Original dataset contributions:', summary['sources'])
print('Distinct classes with training images:', len(class_names))
print('Examples:', class_names[:15])
print('Notes:', *summary['notes'], sep='\n- ')

### Exploratory data analysis: training class frequency

In [ ]:
counts = pd.Series(summary['train_class_counts']).sort_values(ascending=False)
plt.figure(figsize=(13, 5))
counts.plot.bar()
plt.title('Number of labeled training sign crops by class')
plt.xlabel('Class name'); plt.ylabel('Training crop count')
plt.xticks(rotation=90, fontsize=8); plt.tight_layout(); plt.show()
print('Training samples for rarest 10 classes:')
display(counts.tail(10).to_frame('Number of crops'))

### Inspect original full-road images and their ground-truth bounding boxes

The rectangles below are **human annotations from the ZIPs**, not neural network predictions. We use them to check preprocessing before training.

In [ ]:
img_dir = DATA_DIR / 'detection' / 'images' / 'train'
examples = []
for path in sorted(img_dir.glob('*.jpg')):
    lbl = DATA_DIR / 'detection' / 'labels' / 'train' / (path.stem + '.txt')
    if lbl.read_text().strip():
        examples.append((path, lbl))
    if len(examples) == 3: break
fig, axes = plt.subplots(1, len(examples), figsize=(15, 5))
for ax, (path, lbl) in zip(np.atleast_1d(axes), examples):
    with Image.open(path) as inp:
        im = inp.convert('RGB')
    draw = ImageDraw.Draw(im)
    W, H = im.size
    for line in lbl.read_text().splitlines():
        cls, x, y, w, h = map(float, line.split())
        draw.rectangle(((x-w/2)*W, (y-h/2)*H, (x+w/2)*W, (y+h/2)*H), outline='red', width=4)
    ax.imshow(im); ax.set_title(path.stem[:22]); ax.axis('off')
plt.suptitle('Original road images with provided annotations')
plt.tight_layout(); plt.show()

## 3. Tensor preprocessing and loaders

The bounding box is cropped from the full-resolution road image. Training crops use small rotations, random crops and color jitter; validation and test crops are **not augmented**. Horizontal flipping is deliberately avoided because it would reverse arrow directions and create incorrect sign labels.

In [ ]:
from src.classification_data import CropDataset
TRAIN_BATCH = 64
train_ds = CropDataset(DATA_DIR, 'train', train_transform=True)
val_ds = CropDataset(DATA_DIR, 'val')
test_ds = CropDataset(DATA_DIR, 'test')
train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=TRAIN_BATCH, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=TRAIN_BATCH, shuffle=False, num_workers=0)
batch_images, batch_labels = next(iter(train_loader))
print('Train / val / test crops:', len(train_ds), len(val_ds), len(test_ds))
print('Batch tensor shape:', tuple(batch_images.shape), '(B, 3, 64, 64)')
print('Min/max normalized pixel value:', float(batch_images.min()), float(batch_images.max()))
print('Sample class IDs:', batch_labels[:8].tolist())

## 4. Custom CNN architecture (`torch.nn.Module`)

This is a network we define and train ourselves, not a pretrained image classifier. Three pooling operations gradually reduce spatial dimensions, four convolutional blocks learn visual patterns, and a fully connected head outputs **one logit per supported sign category**. `forward()` describes the complete pass from image tensor to class logits.

An identical `TrafficSignCNN` architecture is stored in `src/model.py` so the localhost app can load the saved weights.

In [ ]:
class TrafficSignCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 24, kernel_size=3, padding=1),
            nn.BatchNorm2d(24), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(24, 48, kernel_size=3, padding=1),
            nn.BatchNorm2d(48), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(48, 96, kernel_size=3, padding=1),
            nn.BatchNorm2d(96), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(96, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(p=0.25),
            nn.Linear(128, 128), nn.ReLU(inplace=True),
            nn.Dropout(p=0.25), nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = TrafficSignCNN(num_classes=len(class_names)).to(DEVICE)
print(model)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))
with torch.no_grad():
    print('Output logits shape:', tuple(model(torch.zeros(2, 3, 64, 64).to(DEVICE)).shape))

## 5. Train with CrossEntropyLoss + Adam

To reduce class imbalance, the cross-entropy weights scale approximately with the inverse square root of each class's training frequency, capped at 4. Each epoch reports **train loss, validation loss, train accuracy, validation accuracy, and validation weighted F1**. We select the checkpoint by **validation** F1, never by test metrics.

Training from scratch can take multiple minutes (or longer) on CPU. Run it once, then reuse `models/sign_classifier.pt`.

In [ ]:
CNN_EPOCHS = 15
LEARNING_RATE = 0.001
MODEL_DIR = Path('models'); MODEL_DIR.mkdir(exist_ok=True)
counts_per_class = train_ds.label_counts()
largest = max(counts_per_class.values())
loss_weights = [min(4., (largest / max(1, counts_per_class[i]))**0.5) for i in range(len(class_names))]
loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(loss_weights, device=DEVICE))
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': [], 'val_f1': []}
best_f1 = -1.
for epoch in range(1, CNN_EPOCHS + 1):
    model.train()
    train_loss, train_count, train_correct = 0., 0, 0
    for X, y in train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        predictions = model(X)                      # forward pass
        loss = loss_fn(predictions, y)               # cross-entropy
        loss.backward()                             # backpropagation
        optimizer.step()                            # Adam update
        train_loss += loss.item() * len(y)
        train_correct += (predictions.argmax(1) == y).sum().item()
        train_count += len(y)

    model.eval()
    val_loss, true_labels, pred_labels = 0., [], []
    with torch.inference_mode():
        for X, y in val_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            logits = model(X)
            val_loss += loss_fn(logits, y).item() * len(y)
            true_labels.extend(y.cpu().tolist())
            pred_labels.extend(logits.argmax(1).cpu().tolist())

    tr_loss = train_loss / train_count
    va_loss = val_loss / len(val_ds)
    tr_acc = train_correct / train_count
    va_acc = accuracy_score(true_labels, pred_labels)
    va_f1 = f1_score(true_labels, pred_labels, average='weighted', zero_division=0)
    for key, value in [('train_loss', tr_loss), ('val_loss', va_loss),
                       ('train_accuracy', tr_acc), ('val_accuracy', va_acc), ('val_f1', va_f1)]:
        history[key].append(value)
    print(f'Epoch {epoch:2d}/{CNN_EPOCHS} | train loss={tr_loss:.4f} | val loss={va_loss:.4f} | '
          f'train accuracy={tr_acc:.3f} | val accuracy={va_acc:.3f} | val weighted F1={va_f1:.3f}')
    if va_f1 > best_f1:
        best_f1 = va_f1
        torch.save({'model_state_dict': model.state_dict(), 'class_names': class_names,
                    'image_size': 64, 'epoch': epoch, 'validation_f1': va_f1},
                   MODEL_DIR / 'sign_classifier.pt')

(MODEL_DIR / 'classifier_history.json').write_text(json.dumps(history, indent=2), encoding='utf8')
print('Best classifier saved: models/sign_classifier.pt | best validation weighted F1:', round(best_f1, 4))

## 6. Learning curves (visual confirmation of convergence)

**Interpretation:** validation loss decreasing suggests learning and generalization; rising validation loss with falling training loss suggests overfitting. Compare accuracy and weighted F1 due to class imbalance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(1, len(history['train_loss'])+1), history['train_loss'], label='Train')
axes[0].plot(range(1, len(history['val_loss'])+1), history['val_loss'], label='Validation')
axes[0].set(title='Cross-entropy loss', xlabel='Epoch', ylabel='Loss')
axes[1].plot(range(1, len(history['train_accuracy'])+1), history['train_accuracy'], label='Train accuracy')
axes[1].plot(range(1, len(history['val_accuracy'])+1), history['val_accuracy'], label='Validation accuracy')
axes[1].set(title='Classification accuracy', xlabel='Epoch', ylabel='Accuracy')
for ax in axes: ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'classifier_learning_curves.png', dpi=150)
plt.show()

## 7. Final **independent test** evaluation: CNN accuracy and F1

The test data was reserved before model fitting. **These scores measure sign classification on annotated sign crops, not whether the detector found each sign in the first place.** Class labels missing from training (including GTSDB class ID `100`) are excluded, as reported during preparation.

In [ ]:
checkpoint = torch.load(MODEL_DIR / 'sign_classifier.pt', map_location=DEVICE, weights_only=True)
model.load_state_dict(checkpoint['model_state_dict']); model.eval()
y_true, y_pred = [], []
with torch.inference_mode():
    for X, y in test_loader:
        labels = model(X.to(DEVICE)).argmax(1).cpu().tolist()
        y_true.extend(y.tolist()); y_pred.extend(labels)
final_accuracy = accuracy_score(y_true, y_pred)
final_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
print(f'CNN held-out TEST accuracy: {final_accuracy:.4f}')
print(f'CNN held-out TEST weighted F1: {final_f1:.4f}')
print(f'Test sign crops scored: {len(y_true)}')
print(classification_report(y_true, y_pred,
    labels=list(range(len(class_names))), target_names=class_names, zero_division=0))
(MODEL_DIR / 'classifier_test_metrics.json').write_text(json.dumps({
    'test_accuracy': final_accuracy, 'test_f1_weighted': final_f1,
    'n_test_crops': len(y_true), 'best_val_f1': best_f1}, indent=2))

### Confusion matrix for the most frequent 12 ground-truth test classes

In [ ]:
top_classes = [i for i, n in Counter(y_true).most_common(12)]
filtered_true = [t for t, p in zip(y_true, y_pred) if t in top_classes]
filtered_pred = [p for t, p in zip(y_true, y_pred) if t in top_classes]
mat = confusion_matrix(filtered_true, filtered_pred, labels=top_classes)
fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(mat)
fig.colorbar(im, ax=ax)
ax.set_xticks(range(len(top_classes)), [class_names[i] for i in top_classes], rotation=90)
ax.set_yticks(range(len(top_classes)), [class_names[i] for i in top_classes])
ax.set(xlabel='Predicted class', ylabel='Actual class',
       title='Confusion matrix, 12 frequent actual classes (other predictions omitted)')
plt.tight_layout(); plt.show()
print('NOTE: this plot omits predicted labels outside the 12 selected classes; use full F1 report above.')

## 8. Train a detector for **whole-road images** (bounding boxes)

The custom CNN above predicts a sign's category **after the sign has been found**. To automatically find sign locations on internet photos, we fine-tune a lightweight pretrained YOLO detector on all **5,869 full road images**, using one bounding-box category for signs and the annotated traffic lights. This detector is a separate pretrained component; the CNN above is the model constructed from scratch for the PyTorch assignment.

**Large training job:** A CUDA GPU is strongly recommended; on CPU this can take a long time. The first run may download `yolo11n.pt`. Change `RUN_DETECTOR_TRAINING` to `False` when rerunning the notebook with an existing checkpoint.

In [ ]:
from train_detector import train_detector, evaluate_detector
RUN_DETECTOR_TRAINING = True
DETECTOR_EPOCHS = 35
DETECTOR_IMGSZ = 640
DETECTOR_BATCH = 8   # reduce if GPU runs out of memory

if RUN_DETECTOR_TRAINING:
    train_detector(data_root=DATA_DIR, epochs=DETECTOR_EPOCHS,
                   imgsz=DETECTOR_IMGSZ, batch=DETECTOR_BATCH,
                   device=0 if torch.cuda.is_available() else 'cpu')
else:
    print('Skipped detector training; existing checkpoint required: models/sign_detector.pt')

## 9. Test the detector and plot its training metrics

For object detection, use **precision, recall, mAP@0.5, and mAP@0.5:0.95**, measured on held-out road images. These metrics are different from the CNN classification Accuracy/F1.

In [ ]:
detector_path = MODEL_DIR / 'sign_detector.pt'
if detector_path.is_file():
    det_test = evaluate_detector(data_root=DATA_DIR, checkpoint=detector_path, imgsz=DETECTOR_IMGSZ)
    print('Independent DETECTOR test:', det_test)
else:
    print('Run detector training before measuring detection mAP.')

history_path = Path('runs/traffic_sign_detector/results.csv')
if history_path.is_file():
    detector_history = pd.read_csv(history_path)
    detector_history.columns = [c.strip() for c in detector_history.columns]
    display(detector_history.tail(3))
    available = [c for c in ('metrics/mAP50(B)', 'metrics/mAP50-95(B)') if c in detector_history]
    if available:
        detector_history[available].plot(figsize=(9, 4), title='Validation detector mAP by epoch', grid=True)
        plt.xlabel('Epoch'); plt.ylabel('mAP'); plt.tight_layout(); plt.show()

## 10. Demonstration: Detect + classify signs on an actual road photograph

This is the output you will show in your localhost application: one road image, sign rectangles, and human-readable labels. You can replace `examples/gtsdb_road_example.jpg` with any JPEG or PNG downloaded from the internet.

In [ ]:
from src.inference import load_models, classify_and_annotate
ROAD_IMAGE = Path('examples/gtsdb_road_example.jpg')
if (MODEL_DIR / 'sign_classifier.pt').is_file() and (MODEL_DIR / 'sign_detector.pt').is_file():
    detector, classifier, names, device = load_models()
    original = Image.open(ROAD_IMAGE).convert('RGB')
    marked, predictions = classify_and_annotate(original, detector, classifier, names, device,
                                                detection_threshold=.20, class_threshold=.45)
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    axes[0].imshow(original); axes[0].set_title('Input road image'); axes[0].axis('off')
    axes[1].imshow(marked); axes[1].set_title('AI predictions (not ground truth)'); axes[1].axis('off')
    plt.tight_layout(); plt.show()
    display(pd.DataFrame(predictions))
else:
    print('Train BOTH classifier and detector to enable whole-image predictions.')

## 11. Conclusion & 4-minute defense

**Fill in the measured numbers above after running the notebook** (do not invent metrics):

- Data: **5,869** images from two datasets; GTSDB training further split into train/validation; source test sets held out.
- Input: RGB crops → normalized PyTorch tensors `(B, 3, 64, 64)`; target: sign label. Detector input: full road image; target: bounding-box position.
- Network: 4 convolutional stages with ReLU, BatchNorm, max pooling, dropout, and a dense class head; optimizer Adam, cross-entropy loss.
- **CNN test accuracy:** *insert from Section 7*.
- **CNN test weighted F1:** *insert from Section 7*.
- **Detector test mAP@0.5:** *insert from Section 9*.
- **Limitations:** some classes have only 1–3 training examples; class imbalance and cross-country sign differences affect generalization; two-stage pipeline can fail to find or name a sign.

### Suggested defense timing

| Timing | Demonstrate |
|---|---|
| 0:00–0:35 | Two datasets, sign annotations, class mapping, tensor preprocessing |
| 0:35–1:25 | Your `TrafficSignCNN(nn.Module)` and its `forward` method |
| 1:25–2:25 | Training loop and real loss/accuracy graphs |
| 2:25–3:10 | Test Accuracy/F1 and detection mAP |
| 3:10–4:00 | `streamlit run app.py`: upload a road image and show bounding boxes |

**Model usage:** educational demonstration only — not for real-world driving decisions.